# Chapter 1 — Historical Ciphers

Source: *Cryptography with a Placement Oriented Approach* — Dr. Vikas Srivastava.

This notebook collects every runnable Python listing from Chapter 1 (Historical Ciphers),
organized by section, in the order needed to run top-to-bottom in a fresh kernel.

> Run cells **in order** — later sections reuse functions and variables defined earlier
> (e.g. `normalize`, `letter_to_number`, `number_to_letter`, and the `ciphertext` variable
> used across the guided cryptanalysis examples).

## 1.1 Cryptographic Language and Mathematical Foundations

**Listing 1.1** — Basic alphabet conversion functions, reused throughout the chapter.

In [ ]:
import string

alphabet = string.ascii_uppercase

# Alias used later in the book (Listing 1.7 refers to ALPHABET)
ALPHABET = alphabet


# Convert text to uppercase and remove non-letter characters
def normalize(text):
    text = text.upper()
    result = ""

    for ch in text:
        if ch in alphabet:
            result = result + ch

    return result


# Convert a letter into a number
# A -> 0, B -> 1, ..., Z -> 25
def letter_to_number(ch):
    return ord(ch) - ord('A')


# Convert a number into a letter
# 0 -> A, 1 -> B, ..., 25 -> Z
def number_to_letter(num):
    num = num % 26
    return chr(num + ord('A'))


# Example 1.1
print(normalize("Attack at dawn!"))       # -> ATTACKATDAWN
print(letter_to_number("A"))              # -> 0
print(letter_to_number("T"))              # -> 19

## 1.2 The Shift Cipher

**Listing 1.2** — Python implementation of the Caesar cipher.

In [ ]:
# Caesar encryption
def caesar_encrypt(message, key):

    message = normalize(message)

    ciphertext = ""

    for letter in message:

        number = letter_to_number(letter)

        number = number + key

        cipher_letter = number_to_letter(number)

        ciphertext = ciphertext + cipher_letter

    return ciphertext


# Caesar decryption
def caesar_decrypt(ciphertext, key):

    ciphertext = normalize(ciphertext)

    plaintext = ""

    for letter in ciphertext:

        number = letter_to_number(letter)

        number = number - key

        plain_letter = number_to_letter(number)

        plaintext = plaintext + plain_letter

    return plaintext


# Example
message = "Meet me after class"

key = 7

ciphertext = caesar_encrypt(message, key)

print("Ciphertext :", ciphertext)

plaintext = caesar_decrypt(ciphertext, key)

print("Recovered :", plaintext)

**Listing 1.3** — Exhaustive key search against the Caesar cipher.

In [ ]:
def caesar_bruteforce(ciphertext):
    for key in range(26):
        candidate = caesar_decrypt(ciphertext, key)
        print(f"key={key:2d}: {candidate}")


# Try it on the ciphertext produced above
caesar_bruteforce(ciphertext)

## 1.3 The Affine Cipher

**Listing 1.4** — Extended Euclidean algorithm and modular inverse.

In [ ]:
from math import gcd


def extended_gcd(a: int, b: int):
    if b == 0:
        return a, 1, 0
    g, x1, y1 = extended_gcd(b, a % b)
    return g, y1, x1 - (a // b) * y1


def modular_inverse(a: int, modulus: int = 26) -> int:
    g, x, _ = extended_gcd(a, modulus)
    if g != 1:
        raise ValueError(f"{a} has no inverse modulo {modulus}")
    return x % modulus


# Example: 5 inverse mod 26
print(modular_inverse(5, 26))  # -> 21

**Listing 1.5** — Affine encryption and decryption.

In [ ]:
def affine_encrypt(plaintext: str, a: int, b: int) -> str:
    if gcd(a, 26) != 1:
        raise ValueError("a must be relatively prime to 26")
    text = normalize(plaintext)
    return "".join(
        number_to_letter(a * letter_to_number(ch) + b)
        for ch in text
    )


def affine_decrypt(ciphertext: str, a: int, b: int) -> str:
    if gcd(a, 26) != 1:
        raise ValueError("a must be relatively prime to 26")
    inverse_a = modular_inverse(a, 26)
    text = normalize(ciphertext)
    return "".join(
        number_to_letter(inverse_a * (letter_to_number(ch) - b))
        for ch in text
    )


# Example 1.10: encrypt AFFINE with (a, b) = (5, 8)
enc = affine_encrypt("AFFINE", 5, 8)
print(enc)                              # -> IHHWVC
print(affine_decrypt(enc, 5, 8))        # -> AFFINE

**Listing 1.6** — Exhaustive search over the affine key space.

In [ ]:
def affine_bruteforce(ciphertext: str):
    for a in range(26):
        if gcd(a, 26) != 1:
            continue
        for b in range(26):
            candidate = affine_decrypt(ciphertext, a, b)
            yield a, b, candidate


# Show the first few candidates for the ciphertext "IHHWVC"
for a, b, candidate in affine_bruteforce("IHHWVC"):
    print(f"a={a:2d} b={b:2d}: {candidate}")

## 1.4 The Monoalphabetic Substitution Cipher

**Listing 1.7** — Validation and construction of a substitution key.

In [ ]:
def validate_substitution_key(cipher_alphabet: str) -> str:
    key = normalize(cipher_alphabet)
    if len(key) != 26:
        raise ValueError("The key must contain exactly 26 letters")
    if set(key) != set(ALPHABET):
        raise ValueError("The key must be a permutation of A through Z")
    return key


def make_substitution_maps(cipher_alphabet: str):
    key = validate_substitution_key(cipher_alphabet)
    enc_map = dict(zip(ALPHABET, key))
    dec_map = dict(zip(key, ALPHABET))
    return enc_map, dec_map

**Listing 1.8** — Substitution encryption and decryption.

In [ ]:
def substitution_encrypt(plaintext: str, cipher_alphabet: str) -> str:
    enc_map, _ = make_substitution_maps(cipher_alphabet)
    return "".join(enc_map[ch] for ch in normalize(plaintext))


def substitution_decrypt(ciphertext: str, cipher_alphabet: str) -> str:
    _, dec_map = make_substitution_maps(cipher_alphabet)
    return "".join(dec_map[ch] for ch in normalize(ciphertext))


key = "QWERTYUIOPASDFGHJKLZXCVBNM"
message = "Historical ciphers preserve patterns"
ciphertext = substitution_encrypt(message, key)
recovered = substitution_decrypt(ciphertext, key)

print("Ciphertext:", ciphertext)
print("Recovered :", recovered)

**Listing 1.9** — Counting and sorting ciphertext letters.

> Note: this defines a function named `frequency_table`. Later, Listing 1.12
> (Section 1.6) reuses the same name for a *list* variable — that shadowing
> is intentional in the source text, so run the sections in the order given
> and re-run this cell if you need the function back afterwards.

In [ ]:
from collections import Counter


def frequency_table(text: str):
    clean = normalize(text)
    counts = Counter(clean)
    total = len(clean)
    rows = []
    for letter, count in counts.most_common():
        rows.append((letter, count, count / total))
    return rows


# Try it on the substitution ciphertext produced above
for row in frequency_table(ciphertext):
    print(row)

## 1.5 The Vigenère Cipher

**Listing 1.10** — Vigenère encryption and decryption.

In [ ]:
def vigenere_encrypt(plaintext: str, key: str) -> str:
    text = normalize(plaintext)
    clean_key = normalize(key)
    if not clean_key:
        raise ValueError("The key must contain at least one letter")

    shifts = [letter_to_number(ch) for ch in clean_key]
    result = []
    for index, ch in enumerate(text):
        shift = shifts[index % len(shifts)]
        result.append(number_to_letter(letter_to_number(ch) + shift))
    return "".join(result)


def vigenere_decrypt(ciphertext: str, key: str) -> str:
    text = normalize(ciphertext)
    clean_key = normalize(key)
    if not clean_key:
        raise ValueError("The key must contain at least one letter")

    shifts = [letter_to_number(ch) for ch in clean_key]
    result = []
    for index, ch in enumerate(text):
        shift = shifts[index % len(shifts)]
        result.append(number_to_letter(letter_to_number(ch) - shift))
    return "".join(result)


# Example 1.17: ATTACKATDAWN with key LEMON
enc = vigenere_encrypt("ATTACKATDAWN", "LEMON")
print(enc)                                  # -> LXFOPVEFRNHR
print(vigenere_decrypt(enc, "LEMON"))       # -> ATTACKATDAWN

**Listing 1.11** — Index of coincidence and average column IC.

In [ ]:
from collections import Counter


def index_of_coincidence(text: str) -> float:
    clean = normalize(text)
    n = len(clean)
    if n < 2:
        return 0.0
    counts = Counter(clean)
    numerator = sum(value * (value - 1) for value in counts.values())
    return numerator / (n * (n - 1))


def average_column_ic(ciphertext: str, period: int) -> float:
    clean = normalize(ciphertext)
    columns = [clean[offset::period] for offset in range(period)]
    return sum(index_of_coincidence(col) for col in columns) / period


# Sanity check on the Vigenère ciphertext produced above
print("IC:", index_of_coincidence(enc))
print("Average column IC for period 1..6:")
for period in range(1, 7):
    print(period, average_column_ic(enc, period))

## 1.6 Worked Comparative Examples

### 1.6.1 Example 1 — Identifying a Caesar ciphertext

**Listing 1.12** — Counting ciphertext-letter frequencies.

In [ ]:
ciphertext = (
    "xultpaajcxitltlxaarpjhtiwtgxktghidhipxciwtvgtpilpit"
    "ghlxiwiwtxgqadds"
)

letter_count = {}

for letter in ciphertext:

    if letter in letter_count:

        letter_count[letter] = (
            letter_count[letter] + 1
        )

    else:

        letter_count[letter] = 1


frequency_table = []

for letter in letter_count:

    count = letter_count[letter]

    percentage = (
        100 * count / len(ciphertext)
    )

    frequency_table.append(
        [letter, count, percentage]
    )


def get_count(item):

    return item[1]


frequency_table.sort(
    key=get_count,
    reverse=True
)


print("Letter     Count   Percentage")

for item in frequency_table:

    print(
        item[0],
        "      ",
        item[1],
        "     ",
        round(item[2], 2)
    )

Expected output starts with `t  10  14.93`, `i  9  13.43`, `x  7  10.45`, …
The most frequent ciphertext letter is `t`, hypothesized to map to plaintext `e`,
giving shift key `k = 15`.

**Listing 1.13** — Decrypting the ciphertext using the recovered shift.

In [ ]:
ciphertext = (
    "xultpaajcxitltlxaarpjhtiwtgxktghidhipxciwtvgtpilpit"
    "ghlxiwiwtxgqadds"
)

alphabet = "abcdefghijklmnopqrstuvwxyz"

key = 15

plaintext = ""


for cipher_letter in ciphertext:

    cipher_number = alphabet.index(
        cipher_letter
    )

    plain_number = (
        cipher_number - key
    ) % 26

    plain_letter = alphabet[
        plain_number
    ]

    plaintext = (
        plaintext + plain_letter
    )


print("Recovered text:")

print(plaintext)

**Listing 1.14** — Verifying the recovered shift key.

In [ ]:
plaintext_without_spaces = (
    "ifweallunitewewillcausetheriverstostainthegreat"
    "waterswiththeirblood"
)

alphabet = "abcdefghijklmnopqrstuvwxyz"

key = 15

verified_ciphertext = ""


for plain_letter in plaintext_without_spaces:

    plain_number = alphabet.index(
        plain_letter
    )

    cipher_number = (
        plain_number + key
    ) % 26

    cipher_letter = alphabet[
        cipher_number
    ]

    verified_ciphertext = (
        verified_ciphertext
        + cipher_letter
    )


original_ciphertext = (
    "xultpaajcxitltlxaarpjhtiwtgxktghidhipxciwtvgtpilpit"
    "ghlxiwiwtxgqadds"
)


if verified_ciphertext == original_ciphertext:

    print("Verification successful.")

else:

    print("Verification failed.")

The recovered plaintext is attributed to Tecumseh's Speech to the Osages.

### 1.6.3 Example 3 — Substitution from structure

**Listing 1.15** — Calculating ciphertext-letter frequencies for a monoalphabetic
substitution ciphertext (this redefines `ciphertext`, which the next few listings reuse).

In [ ]:
ciphertext = """
lrvmnir bpr sumvbwvr jx bpr lmiwv yjeryrkbi jx qmbm wi
bpr xjvni mkd ymibrut jx irhx wi bpr riirkvr jx
ymbinlmtmipw utn qmumbr dj w ipmhh but bj rhnvwdmbr bpr
yjeryrkbi jx bpr qmbm mvvjudwko bj yt wkbrusurbmbwjk
lmird jk xjubt trmui jx ibndt
wb wi kjb mk rmit bmiq bj rashmwk rmvp yjeryrkb mkd wbi
iwokwxwvmkvr mkd ijyr ynib urymwk nkrashmwkrd bj ower m
vjyshrbr rashmkmbwjk jkr cjnhd pmer bj lr fnmhwxwrd mkd
wkiswurd bj invp mk rabrkb bpmb pr vjnhd urmvp bpr ibmbr
jx rkhwopbrkrd ywkd vmsmlhr jx urvjokwgwko ijnkdhrii
ijnkd mkd ipmsrhrii ipmsr w dj kjb drry ytirhx bpr xwkmh
mnbpjuwbt lnb yt rasruwrkvr cwbp qmbm pmi hrxb kj djnlb
bpmb bpr xjhhjcwko wi bpr sujsru msshwvmbwjk mkd
wkbrusurbmbwjk w jxxru yt bprjuwri wk bpr pjsr bpmb bpr
riirkvr jx jqwkmcmk qmumbr cwhh urymwk wkbmvb
"""

letters_only = ""

for character in ciphertext:

    if character.isalpha():

        letters_only = (
            letters_only
            + character.lower()
        )


letter_count = {}

for letter in letters_only:

    if letter in letter_count:

        letter_count[letter] = (
            letter_count[letter] + 1
        )

    else:

        letter_count[letter] = 1


frequency_table = []

for letter in letter_count:

    count = letter_count[letter]

    relative_frequency = (
        count / len(letters_only)
    )

    frequency_table.append(
        [letter, count, relative_frequency]
    )


def get_count(item):

    return item[1]


frequency_table.sort(
    key=get_count,
    reverse=True
)


print("Total letters:", len(letters_only))

print("\nLetter      Count     Relative frequency")

for item in frequency_table:

    print(
        item[0],
        "      ",
        item[1],
        "     ",
        round(item[2], 4)
    )

Working through frequency analysis and short-word guesses (e.g. `lrvmnir` &rarr;
`because`) progressively yields the mapping `r&rarr;e, b&rarr;t, m&rarr;a, p&rarr;h, j&rarr;o,
x&rarr;f, w&rarr;i, i&rarr;s, k&rarr;n, d&rarr;d`.

**Listing 1.16** — Producing a partial plaintext from the substitutions found so far.

In [ ]:
partial_key = {
    "r": "e",
    "b": "t",
    "m": "a",
    "p": "h",
    "j": "o",
    "x": "f",
    "w": "i",
    "i": "s",
    "k": "n",
    "d": "d"
}


partial_plaintext = ""

for character in ciphertext:

    if character in partial_key:

        partial_plaintext = (
            partial_plaintext
            + partial_key[character]
        )

    elif character.isalpha():

        partial_plaintext = (
            partial_plaintext + "_"
        )

    else:

        partial_plaintext = (
            partial_plaintext + character
        )


print(partial_plaintext)

**Listing 1.17** — The fully recovered ciphertext-to-plaintext substitution
(after continuing the manual cryptanalysis in the book).

In [ ]:
decryption_key = {
    "a": "x",
    "b": "t",
    "c": "w",
    "d": "d",
    "e": "v",
    "f": "q",
    "g": "z",
    "h": "l",
    "i": "s",
    "j": "o",
    "k": "n",
    "l": "b",
    "m": "a",
    "n": "u",
    "o": "g",
    "p": "h",
    "q": "k",
    "r": "e",
    "s": "p",
    "t": "y",
    "u": "r",
    "v": "c",
    "w": "i",
    "x": "f",
    "y": "m"
}

Note: the ciphertext symbols `z` and `j` (as a *plaintext* letter) do not occur
in this short message, so their mapping cannot be recovered from it alone.

**Listing 1.18** — Decrypting the complete substitution ciphertext.

In [ ]:
plaintext = ""

for character in ciphertext:

    if character in decryption_key:

        plaintext = (
            plaintext
            + decryption_key[character]
        )

    else:

        plaintext = (
            plaintext + character
        )


print(plaintext)

This recovers the full plaintext passage discussed in the book (Section 1.6.3),
completing the guided cryptanalysis example.

---

*End of Chapter 1 code listings.* Chapters 2–4 of the source book (Principles of
Modern Cryptography, Stream Ciphers, Worked Out Problems) contain no separately
labeled Python listings — they are theory-only in this edition.